# CIDS · 03 — S2a: SAM-Med3D-turbo fine-tuned on images only (Exp 3 baseline, variant A)

No text, no frequency prompts. The image encoder stays frozen (we reuse the S0 embeddings); only the **mask decoder** is adapted with **LoRA** (r=8 on q/v, ~71k parameters) + 2 learnable box-corner tokens (turbo has no 3D box prompt).

**Settings:** GPU T4 x2 (one GPU used) · **Internet ON** (downloads the turbo checkpoint) · Persistence: Files.  
**Inputs (Add Input):** `cids-sam-emb` (S0 output) **and** the 2-channel cache `mama-mia-preprocessed-cache-2ch` (for full-resolution Dice/HD95/NSD).  
**Split:** `configs/splits.csv` — train 1080 / val 120 (best epoch) / test 306 (once, at the end).

**Train prompts:** jittered GT box (50%) or a random click, then up to 4 corrective clicks.  
**Test:** zero-shot SAM vs fine-tuned SAM with the *same* seeded clicks — crop Dice for 1–5 clicks and box(+clicks), plus full-resolution metrics at 1 mm.

**Run:** smoke test first, then *Save Version → Save & Run All* (~1–2 h).

In [ ]:
TUNE = 'lora'               # 'lora' (variant A, default)  or  'decoder_full' (whole mask decoder, 7.6M params)

REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

EMB_ROOT   = ''               # '' = auto-detect cids-sam-emb (folder with index.csv, train/, test/)
CACHE_ROOT = ''               # '' = auto-detect the 2-channel cache (train/ + val/)
RESUME_FROM = ''              # path to last.pt from a previous version's output, if continuing
EPOCHS = 60

CODE_DIR = '/kaggle/working/cids'
OUT      = f'/kaggle/working/s2a_{TUNE}'

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q medim einops appdirs
!pytest -q tests

In [ ]:
import sys, torch, pandas as pd
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
p = torch.cuda.get_device_properties(0); print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')
sys.path.insert(0, CODE_DIR)
sys.path.insert(0, f'{CODE_DIR}/scripts')
from s2a_sam_finetune import find_emb_root
from cids.data.cache import find_cache_root
EMB_ROOT = str(find_emb_root(EMB_ROOT))
n_tr, n_te = len(list(Path(EMB_ROOT, 'train').glob('*.npz'))), len(list(Path(EMB_ROOT, 'test').glob('*.npz')))
print('embeddings:', EMB_ROOT, '|', n_tr, 'train /', n_te, 'test'); assert (n_tr, n_te) == (1200, 306)
try:
    CACHE_ROOT = str(find_cache_root(CACHE_ROOT)); print('cache:', CACHE_ROOT)
except FileNotFoundError:
    CACHE_ROOT = 'none'; print('!! 2-channel cache not attached -> only crop-level Dice will be reported')
sp = pd.read_csv('configs/splits.csv'); print(sp.split.value_counts().to_dict())
print('tune:', TUNE, '| out:', OUT)

## Smoke test — 16 train, 1 epoch, 8 val + 8 test (~3 min)
Check: no errors, loss is a normal number, and the **zero_shot click1** test Dice is clearly above 0 (around 0.4–0.6). Near-zero means the checkpoint did not load — stop and send me the output.

In [ ]:
!python scripts/s2a_sam_finetune.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --tune {TUNE} \
    --out /kaggle/working/s2a_smoke --epochs 1 --val_every 1 --limit_train 16 --limit_eval 8
!rm -rf /kaggle/working/s2a_smoke

## Full run (~1–2 h) — use *Save Version → Save & Run All*

In [ ]:
resume = f'--resume_from {RESUME_FROM}' if RESUME_FROM else ''
!python scripts/s2a_sam_finetune.py --emb_root {EMB_ROOT} --cache_root {CACHE_ROOT} --tune {TUNE} \
    --out {OUT} --epochs {EPOCHS} {resume}

In [ ]:
import json, matplotlib.pyplot as plt
log = pd.read_csv(f'{OUT}/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(log.epoch, log.loss); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
v = log.dropna(subset=['val_score'])
for c in ['val_click1', 'val_click5', 'val_box1']:
    ax[1].plot(v.epoch, v[c], marker='o', label=c[4:])
ax[1].legend(); ax[1].set_title('val Dice (120, crop)'); plt.tight_layout(); plt.show()
print(json.dumps(json.load(open(f'{OUT}/test_summary.json')), indent=2))
!du -sh {OUT}